In [15]:
import torch
import pandas as pd
import os
import sys
sys.path.append('../')

In [16]:
def count_parameters(path: str):
    model: dict = torch.load(os.path.join(path, "model.pth"), map_location='cpu', weights_only=True)
    num_params = 0
    num_params_non_zero = 0
    for k, v in model.items():
        num_params += v.numel()
        num_params_non_zero += (v != 0).sum().int().item()
    return int(num_params_non_zero)

In [17]:
models = [
    ("/local/scratch/clmn1/videoNCA/CholecDataset/dashing-wildflower-38", "NCA"),
    #("/local/scratch/clmn1/videoNCA/CholecDataset/wild-fire-10", "NCA"),
    ("/local/scratch/clmn1/videoNCA/CholecDataset/spring-star-14", "SegFormer"),
    ("/local/scratch/clmn1/videoNCA/CholecDataset/devoted-glade-13", "UNet"),
    ("/local/scratch/clmn1/videoNCA/CholecDataset/dutiful-resonance-15", "SwinUNetv2")
]
df = []
for model in models:
    dices = 100 * pd.read_csv(os.path.join(model[0], "dices_val.csv"))
    maDice = dices.mean().mean()
    maDice_std = dices.mean().std()
    miDice = dices.stack().mean()
    miDice_std = dices.stack().std()
    dices = dices.mean()
    dices.name = model[1]
    dices["maDice"] = maDice
    dices["maDice std"] = maDice_std
    dices["miDice"] = miDice
    dices["miDice std"] = miDice_std
    dices["num params"] = count_parameters(model[0])
    df.append(dices)
df = pd.DataFrame(df)
df["num params"] = df["num params"].astype(int)
df

,abdominal_wall (1),liver (2),gastrointestinal_tract (3),fat (4),grasper (5),connective_tissue (6),blood (7),cystic_duct (8),hook (9),gallbladder (10),maDice,maDice std,miDice,miDice std,num params
NCA,76.820867,86.920496,59.698945,84.283668,79.999692,76.196303,53.613402,3.728954e+01,87.569025,66.753736,70.914568,16.434008,76.624660,21.850488,27465
SegFormer,82.678384,91.939547,63.263710,86.209705,82.573968,81.535813,48.369170,1.422813e-07,91.071123,85.039063,71.268048,28.408481,81.714389,22.090994,3717484
UNet,44.566106,73.538938,32.474735,67.228684,55.698150,62.953309,3.499513,1.422813e-07,61.773551,42.592907,44.432589,25.676718,53.581915,25.153751,68331670
SwinUNetv2,80.423848,89.969177,56.056215,84.633231,78.571277,77.217639,41.421908,1.422813e-07,85.037425,80.396803,67.372752,27.935336,77.957869,23.207100,27941028


In [18]:
df_latex = df[["maDice", "maDice std", "miDice", "miDice std", "num params"]].copy()
for metric in ["maDice", "miDice"]:
    mean_col = metric
    std_col = metric + " std"
    df_latex[metric] = (
                df_latex[mean_col].map("{:.1f}".format)
                + " $\\pm$ "
                + df_latex[std_col].map("{:.1f}".format)
            )
    df_latex = df_latex.drop(columns=[std_col])

In [19]:
df_latex

,maDice,miDice,num params
NCA,70.9 $\pm$ 16.4,76.6 $\pm$ 21.9,27465
SegFormer,71.3 $\pm$ 28.4,81.7 $\pm$ 22.1,3717484
UNet,44.4 $\pm$ 25.7,53.6 $\pm$ 25.2,68331670
SwinUNetv2,67.4 $\pm$ 27.9,78.0 $\pm$ 23.2,27941028


In [20]:
formatters =({
    "num params": lambda x: f"{int(x):,}"
})

df_latex = df_latex.to_latex(formatters=formatters)

print(df_latex)


\begin{tabular}{lllr}
\toprule
 & maDice & miDice & num params \\
\midrule
NCA & 70.9 $\pm$ 16.4 & 76.6 $\pm$ 21.9 & 27,465 \\
SegFormer & 71.3 $\pm$ 28.4 & 81.7 $\pm$ 22.1 & 3,717,484 \\
UNet & 44.4 $\pm$ 25.7 & 53.6 $\pm$ 25.2 & 68,331,670 \\
SwinUNetv2 & 67.4 $\pm$ 27.9 & 78.0 $\pm$ 23.2 & 27,941,028 \\
\bottomrule
\end{tabular}

